###  MicroGrad demo

Based on: https://github.com/karpathy/micrograd

In [ ]:
import random
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
%matplotlib inline

In [ ]:
from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

In [ ]:
np.random.seed(42)
random.seed(42)

In [ ]:
# make up a dataset

from sklearn.datasets import make_moons, make_blobs
X, y = make_moons(n_samples=100, noise=0.1)

# y = y*2 - 1 # make y as -1 or 1
y = y*4 - 2 # make y as -2 or 2
# visualize in 2D
plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], c=y, s=20, cmap='jet')

In [ ]:
# f'(x) = 3 k3 x2 + 2 k2 x + k1
# ničle: (-k2 +- koren (k2 ^2 - 4 k3 * k1)) / 2 k3
# (-k2 - koren (k2 ^2 - 4 k3 * k1)) / 2 k3 = 0 => k2 = - koren (k2 ^2 - 4 k3 * k1))
# (-k2 + koren (k2 ^2 - 4 k3 * k1)) / 2 k3 = 1 => k2 = + koren (k2 ^2 - 4 k3 * k1)) - 2 k3
# koren (k2 ^2 - 4 k3 * k1)) = k3
# k3 = 1
# k2 ^2 - 4 k1 = 1 => k2 ^2 = 4 k1 + 1 => k1 = 2, k2 = 3

# k3 = 3/4; k2 = -43/44; k1 = -5/8; k0 = 7/10

k3=1.029; k2=-1.337; k1=-0.605; k0=0.686

x = 1
print(3 * k3 * x ** 2 + 2 * k2 * x + k1)

# f(-1) = -0.5
# f(0) = 0.7
# f(1) = -0.25
# f(2) = > 1
# -1, -0.5; 0, 0.7, 1, -0.25

def funsep(k3, k2, k1, k0, x):
    tmp = k3 * x ** 3 + k2 * x ** 2 + k1 * x + k0
    return np.clip(tmp, a_min=-0.7, a_max=1.3)

x = np.arange(-1.5, 2.5, 0.05)
z = funsep(k3, k2, k1, k0, x)
plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], c=y, s=20, cmap='jet')
plt.plot(x, z)

In [ ]:
print(-0.5, funsep(k3, k2, k1, k0, -0.5), 0.5)
print(0, funsep(k3, k2, k1, k0, 0), 0.73)
print(1, funsep(k3, k2, k1, k0, 1), -0.25)
print(1.5, funsep(k3, k2, k1, k0, 1.5), 0.25)
X3 = [-0.5, 0.1, 1.0, 1.5]
y3 = [0.5, 0.70, -0.25, 0.25]

In [ ]:
from sklearn.utils import check_random_state

def make_heart(n_samples=500, noise=0.03, random_state=None):
    rng = check_random_state(random_state)
    X = rng.uniform(-1.5, 1.5, size=(n_samples*4, 2))
    f = (X[:,0]**2 + X[:,1]**2 - 1)**3 - (X[:,0]**2)*(X[:,1]**3)
    y = (f <= 0).astype(int)  # 1=inside heart, 0=outside
    # balance classes
    idx_in = np.where(y==1)[0][:n_samples//2]
    idx_out = np.where(y==0)[0][:n_samples - len(idx_in)]
    idx = np.concatenate([idx_in, idx_out])
    X, y = X[idx], y[idx]
    X += rng.normal(scale=noise, size=X.shape)
    return X, y

# X1, y1 = make_moons(n_samples=400, noise=0.15)
X, y = make_heart(n_samples=100, noise=0.03, random_state=42)

# y = y*2 - 1 # make y as -1 or 1
y = y*4 - 2 # make y as -2 or 2
# visualize in 2D
plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], c=y, s=20, cmap='jet')

In [ ]:
# display generated data
print(len(X))
list(zip(X, y))

In [ ]:
# initialize a model 
# model = MLP(2, [16, 16, 1]) # 2-layer neural network
model = MLP(2, [7, 7, 1]) # 2-layer neural network
print(model)
print("number of parameters", len(model.parameters()))

In [ ]:
# loss function
def loss(batch_size=None):
    
    # inline DataLoader :)
    if batch_size is None:
        Xb, yb = X, y
    else:
        ri = np.random.permutation(X.shape[0])[:batch_size]
        Xb, yb = X[ri], y[ri]
    inputs = [list(map(Value, xrow)) for xrow in Xb]
    
    # forward the model to get scores
    scores = list(map(model, inputs))
    
    # svm "max-margin" loss
    losses = [(1 + -yi*scorei).relu() for yi, scorei in zip(yb, scores)]
    data_loss = sum(losses) * (1.0 / len(losses))
    # L2 regularization
    alpha = 1e-4
    reg_loss = alpha * sum((p*p for p in model.parameters()))
    total_loss = data_loss + reg_loss
    
    # also get accuracy
    accuracy = [(yi > 0) == (scorei.data > 0) for yi, scorei in zip(yb, scores)]
    return total_loss, sum(accuracy) / len(accuracy)

total_loss, acc = loss()
print(total_loss, acc)

In [ ]:
# optimization
def optimize(learning_rate, steps = 1):
    for k in range(steps):
        
        # forward
        total_loss, acc = loss()
        
        # backward
        model.zero_grad()
        total_loss.backward()
        
        # update (sgd)
        # learning_rate = 1.0 - 0.95*(epoch+k)/max_epoch_and_steps
        # if learning_rate < 0.01:
        #     learning_rate = 0.01
        for p in model.parameters():
            p.data -= learning_rate * p.grad
        
        total_loss, acc = loss() # for correct display after the total_loss.backward()
        if k % 1 == 0:
            print(f"step {k} loss {total_loss.data}, accuracy {acc*100}%")
    return total_loss.data, acc


In [ ]:
# visualize decision boundary - obsolete
def visualize():
    h = 0.25
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                        np.arange(y_min, y_max, h))
    Xmesh = np.c_[xx.ravel(), yy.ravel()]
    inputs = [list(map(Value, xrow)) for xrow in Xmesh]
    scores = list(map(model, inputs))
    Z = np.array([s.data > 0 for s in scores])
    Z = Z.reshape(xx.shape)

    fig = plt.figure()
    plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
    # plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.Spectral)

    myinputs = [list(map(Value, xrow)) for xrow in X]
    myscores = list(map(model, myinputs))
    accurate = [(yi > 0) == (scorei.data > 0) for yi, scorei in zip(y, myscores)]
    myy = y.copy()
    for i in range(len(myy)):
        if not accurate[i]:
            if y[i] > 0:
                myy[i] = 1
            else:
                myy[i] = -1
    print(sum(accurate))

    plt.scatter(X[:, 0], X[:, 1], c=myy, s=20, cmap=plt.cm.Spectral)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())


In [ ]:
visualize()

In [ ]:
# optimize(0.5)
# visualize()

In [ ]:
# optimize(0.4)
# visualize()

In [ ]:
# optimize(0.3, 3)
# visualize()

In [ ]:
# optimize(0.1, 10)
# visualize()

In [ ]:
# new frame visualization
def visualize_frame(ax, mytitle):
    h = 0.25
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Xmesh = np.c_[xx.ravel(), yy.ravel()]
    inputs = [list(map(Value, xrow)) for xrow in Xmesh]
    scores = list(map(model, inputs))
    Z = np.array([s.data > 0 for s in scores]).reshape(xx.shape)

    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())

    # draw and COLLECT the artists for this frame
    cset = ax.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
    myinputs = [list(map(Value, xrow)) for xrow in X]
    myscores = list(map(model, myinputs))
    accurate = [(yi > 0) == (scorei.data > 0) for yi, scorei in zip(y, myscores)]
    myy = y.copy()
    for i in range(len(myy)):
        if not accurate[i]:
            if y[i] > 0:
                myy[i] = 1
            else:
                myy[i] = -1
    sc = ax.scatter(X[:, 0], X[:, 1], c=myy, s=20, cmap=plt.cm.Spectral)
    title = ax.text(0.5, 1.02, mytitle, transform=ax.transAxes,
                    ha="center", fontsize=12)

    # Return a list of artists (wrap cset; matplotlib>=3.10 has no .collections)
    return [cset, sc, title]


In [ ]:
# build plots and visialize gifs
num_epochs = 51
lr = [] # learning rates between 1 and 0.1
for i in range(num_epochs):
    lr.append(1.0 - i*0.9/num_epochs)

fig, ax = plt.subplots(figsize=(6,6))
ims = []

# initial frame
ims.append(visualize_frame(ax, 'Initial'))

# training loop (call after each update)
for epoch in range(num_epochs):
    # ... train / update model ...
    print(f"Epoch {epoch}/{num_epochs-1}")
    myloss, myacc = optimize(lr[epoch], 1)
    ims.append(visualize_frame(ax, f"Epoch {epoch} acc= {myacc*100:.1f}"))

ani = animation.ArtistAnimation(fig, ims, interval=1500, blit=False, repeat=False, repeat_delay=7000)
ani.save("77_decision_boundary.gif", writer="pillow")
plt.close(fig)


In [ ]:
model.parameters

In [ ]:
from graphviz import Digraph

def visualize_mlp_architecture(mlp, input_labels=None, filename="mlp_architecture", fmt="svg"):
    """
    Draws the network graph: inputs -> layers -> outputs.
    Edge labels are weights; neuron boxes show bias.
    Works with Karpathy's micrograd.nn MLP/Layer/Neuron.
    """
    dot = Digraph(format=fmt, graph_attr={'rankdir': 'LR', 'splines': 'true'})

    # Infer input size and labels
    nin = len(mlp.layers[0].neurons[0].w)
    if input_labels is None:
        input_labels = [f"x{i}" for i in range(nin)]

    # --- Inputs (rank=same) ---
    with dot.subgraph(name="cluster_inputs") as c:
        c.attr(label="Inputs", color="grey")
        for i, lbl in enumerate(input_labels):
            c.node(f"in_{i}", lbl, shape="circle")

    prev_nodes = [f"in_{i}" for i in range(nin)]

    # --- Hidden/Output layers ---
    for li, layer in enumerate(mlp.layers):
        with dot.subgraph(name=f"cluster_L{li}") as c:
            c.attr(label=f"Layer {li}", color="lightgrey")
            layer_nodes = []
            for ni, neuron in enumerate(layer.neurons):
                bias = getattr(neuron.b, "data", neuron.b)
                node_id = f"L{li}N{ni}"
                c.node(node_id, f"N{ni}\n(bias={bias:.3f})", shape="box", style="rounded")
                layer_nodes.append(node_id)

                # Connect from previous layer inputs with weight labels
                for pi, pnode in enumerate(prev_nodes):
                    w = getattr(neuron.w[pi], "data", neuron.w[pi])
                    # Optional styling: dashed for negative weights
                    style = "solid" if w >= 0 else "dashed"
                    dot.edge(pnode, node_id, label=f"{w:.3f}", style=style)

        prev_nodes = layer_nodes

    # Mark final layer as Outputs
    dot.node("out_label", "Output(s)", shape="plaintext")
    for n in prev_nodes:
        dot.edge(n, "out_label")

    # Render
    path = dot.render(filename=filename, cleanup=True)
    print(f"Saved: {path}")
    return path


In [ ]:
visualize_mlp_architecture(model, filename="mlp_architecture")

In [ ]:
# simple one
def summarize_mlp(mlp):
    for li, layer in enumerate(mlp.layers):
        print(f"Layer {li}: {len(layer.neurons)} neurons")
        for ni, n in enumerate(layer.neurons):
            w = [float(wi.data) for wi in n.w]
            b = float(n.b.data)
            print(f"  Neuron {ni}: w={w}, b={b}")

summarize_mlp(model)


In [ ]:
# more complex, including inputs, hidden layers (weights, biases, activation function), and outputs
def summarize_mlp(mlp, input_labels=None):
    """
    Prints a detailed summary of a micrograd MLP model,
    including the input layer, hidden layers, and output layer.

    Args:
        model: micrograd.nn.MLP instance
        input_labels: optional list of names for input features
    """
    # Infer number of inputs
    n_inputs = len(mlp.layers[0].neurons[0].w)
    if input_labels is None:
        input_labels = [f"x{i}" for i in range(n_inputs)]

    print("=== MLP Architecture Summary ===")
    print(f"Input layer: {n_inputs} inputs → {input_labels}")
    print("-" * 40)

    for li, layer in enumerate(mlp.layers):
        n_neurons = len(layer.neurons)
        print(f"Layer {li + 1}: {n_neurons} neuron(s)")
        for ni, neuron in enumerate(layer.neurons):
            # Extract numeric values for weights and bias
            weights = [float(wi.data) if hasattr(wi, "data") else float(wi)
                       for wi in neuron.w]
            bias = float(neuron.b.data) if hasattr(neuron.b, "data") else float(neuron.b)

            # Make readable weight-label mapping
            weight_str = ", ".join(f"w{lbl}={w:.3f}" for lbl, w in zip(input_labels, weights))
            print(f"  Neuron {ni}: ({weight_str}), bias={bias:.3f}; {neuron}")

        print("-" * 40)
        # Update input labels for next layer
        input_labels = [f"L{li+1}N{j}" for j in range(n_neurons)]

    print("Output layer size:", len(mlp.layers[-1].neurons))
    print("================================")

summarize_mlp(model)

### Issues for discussion:

* **NN architecture** (inputs, hidden layers, neurons in each hidden layer, ouputs)
* **Data split**: training 80%, validation 10%, testing 10%
* **Forward pass execution**: the input data moves step-by-step through all the layers of a neural network to produce an output prediction
* **Number of epochs for training**: how many times the neural network sees and learns from the entire training dataset during training
* **Batch size for training**: the number of training examples the neural network looks at before updating its weights once
* **Loss function** (classification error + regularization): combines how wrong the model’s predictions are (classification error) with a penalty for being too complex (regularization) to help it learn accurately without overfitting
* **Learning rate**: controls how big a step the neural network takes when updating its weights to reduce the loss after each batch

### Separation with a cubic equation

$$
f(x) = ax^3 + bx^2 + cx + d 
$$

In [ ]:
# data from X, y = make_moons(n_samples=100, noise=0.1)
for i in range(len(X3)):
    print("X3:", i, X3[i], y3[i])
for i in range(len(X)):
    print("X:", i, X[i], y[i])
# y is negative(-2) or positive (2)

In [ ]:
from graphviz import Digraph

class Value0:
    """ stores a single scalar value and its gradient """

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0
        # internal variables used for autograd graph construction
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op # the op that produced this node, for graphviz / debugging / etc
        self.label = label

    def __add__(self, other):
        other = other if isinstance(other, Value0) else Value0(other)
        out = Value0(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value0) else Value0(other)
        out = Value0(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value0(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        out = Value0(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value0(t, (self, ), 'tanh')
    
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
    
        return out
        
    def backward(self):

        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __neg__(self): # -self
        return self * -1

    def __radd__(self, other): # other + self
        return self + other

    def __sub__(self, other): # self - other
        return self + (-other)

    def __rsub__(self, other): # other - self
        return other + (-self)

    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __rtruediv__(self, other): # other / self
        return other * self**-1

    def __repr__(self):
        return f"Value0(data={self.data}, grad={self.grad})"

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [ ]:
a = Value0(3/4, label = "a")
b = Value0(-43/44, label = "b")
c = Value0(-5/8, label = "c")
d = Value0(7/10, label = "d")

a = Value0(1.0, label = "a")
b = Value0(-1.0, label = "b")
c = Value0(-1.0, label = "c")
d = Value0(1.0, label = "d")

inputs0 = [Value0(xr0) for xr0 in X3]
# inputs1 = [Value0(xr1) for _ , xr1 in X]

outputs0 = [Value0(y0) for y0 in y3]

In [ ]:
max_epochs = 501
learning_rate = 0.1
for epoch in range(max_epochs):
    # print(f"Step_in {epoch}: a={a.data:.3f}, b={b.data:.3f}, c={c.data:.3f}, d={d.data:.3f}")

    outputs1 = [(a * x0 ** 3 + b * x0 ** 2 + c * x0 + d) for x0 in inputs0]

    # ((inputs1 - outputs1) * outputs0).relu()
    # + točka nad funkcijo    + 2 error
    # +                       - 2 OK
    # - točka pod funkcijo    + 2 OK
    # -                       - 2 error
    offset = Value0(0.0) 
    losses = [(offset + (out0 - out1) ** 2) for out0, out1 in zip(outputs0, outputs1)]
    # losses = [(offset + (in1 - out1) * out0) for in1, out1, out0 in zip(inputs1, outputs1, outputs0)]
    data_loss = sum(losses) * (1.0 / len(losses))
    # L2 regularization
    alpha = 1e-3
    reg_loss = alpha * (a*a + b*b + c*c + d*d)
    total_loss = data_loss + reg_loss

    accuracy = 0.0
    acc = 0.0
    # accuracy = [(ls.data < offset.data) for ls in losses]
    # acc = sum(accuracy) / len(accuracy)
    # for myi in range(len(X3)):
    #     print(f"{myi}: x:{inputs0[myi].data:.3f}: y:{outputs0[myi].data}, f:{outputs1[myi].data:.3f} l:{losses[myi].data:.3f}")

    total_loss.backward()
    if epoch % 10 == 0:
        print(f"Epoch: {epoch}, loss: {total_loss.data:.3f}, accuracy: {acc*100.0:.1f}")
    # print(f"Grads: a={a.grad:.3f}, b={b.grad:.3f}, c={c.grad:.3f}, d={d.grad:.3f}")

    # Update parameters (gradient descent)
    a.data -= learning_rate * a.grad
    b.data -= learning_rate * b.grad
    c.data -= learning_rate * c.grad
    d.data -= learning_rate * d.grad
    # print(f"Step_out {epoch}: a={a.data:.3f}, b={b.data:.3f}, c={c.data:.3f}, d={d.data:.3f}")
    # print()

    # Reset gradients
    a.grad = 0
    b.grad = 0
    c.grad = 0
    d.grad = 0

print(f"Trained: a={a.data:.3f}, b={b.data:.3f}, c={c.data:.3f}, d={d.data:.3f}")



In [ ]:
k3 = a.data
k2 = b.data
k1 = c.data
k0 = d.data
x = 1
print(3 * k3 * x ** 2 + 2 * k2 * x + k1)

def funsep(k3, k2, k1, k0, x):
    tmp = k3 * x ** 3 + k2 * x ** 2 + k1 * x + k0
    return np.clip(tmp, a_min=-0.7, a_max=1.3)

x = np.arange(-1.5, 2.5, 0.05)
z = funsep(k3, k2, k1, k0, x)
plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], c=y, s=20, cmap='jet')
plt.plot(x, z)

In [ ]:
# linear regression example
# 
from micrograd.engine import Value

# Training data (x, y)
# xs = [0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0]
# ys = [0.0, 5.0, 7.0, 15.0, 10.0, 21.0, 23.0, 29.0, 31.0]

# Create some input / output data
xs = [0.03, 0.19, 0.34, 0.46, 0.78, 0.81, 1.08, 1.18, 1.39, 1.60, 1.65, 1.90]
ys = [0.67, 0.85, 1.05, 1.0, 1.40, 1.5, 1.3, 1.54, 1.55, 1.68, 1.73, 1.6 ]

print(xs)
print(ys)

# Initialize parameters
w = Value(-0.5)
b = Value(2.0)

# Create figure and axis
fig, ax = plt.subplots()
ims = []   # list to store frames
# ax.scatter([0.0], [100.0], c='black', s=20, cmap=plt.cm.Spectral)
# ax.scatter([0.0], [-100.0], c='black', s=20, cmap=plt.cm.Spectral)
ax.scatter(xs, ys, c='red', s=20, cmap=plt.cm.Spectral)

max_epochs = 101

# Training loop
for epoch in range(max_epochs):
    # print(f"Step_in {epoch}: w={w.data:.3f}, b={b.data:.3f}")
    # Forward pass: y_pred = w*x + b
    y_pred = [w * x + b for x in xs]
    loss = sum((yout - y) ** 2 for yout, y in zip(y_pred, ys)) # + w ** 2 + n ** 2 # regularizacija

    k = w.data  # slope
    n = b.data   # intercept

    # Generate x-values
    x = np.linspace(0, 2, 10)  # from -10 to 10

    # Compute y-values
    y = k * x + n
    y = np.clip(y, 0.6, 2.0)
    # y = np.clip(y, -100.0, 100.0)

    if epoch % 1 == 0:
        frame = ax.plot(x, y, color='green', label=f'y = {k}x + {n}')
        title = ax.text(0.5, 1.02, f"Regression ({epoch} / {max_epochs-1} epochs)",
                        transform=ax.transAxes, ha='center', fontsize=12)
        ims.append(frame + [title])

    # print('Original:', ys)
    # print('Predicted:', y_pred)

    # Backward pass
    loss.backward()
    print('Loss:', loss)

    # Update parameters (gradient descent)
    w.data -= 0.01 * w.grad
    b.data -= 0.01 * b.grad
    print(f"Step_out {epoch}: w={w.data:.3f}, b={b.data:.3f}")

    # Reset gradients
    w.grad = 0
    b.grad = 0

print(f"Trained: w={w.data:.3f}, b={b.data:.3f}")

# Create and save the animation
ani = animation.ArtistAnimation(fig, ims, interval=200, blit=True)
ani.save("linear_regression.gif", writer='pillow')
plt.close(fig)

In [ ]:
fig = plt.figure()
plt.scatter(xs, ys, c='red', s=20, cmap=plt.cm.Spectral)

# Given parameters
k = w.data  # slope
n = b.data   # intercept
# k = 2.0  # slope
# n = 0.0   # intercept

# Generate x-values
x = np.linspace(0, 2, 10)  # from -10 to 10

k = 0.531
n = 0.815

# Compute y-values
y = k * x + n

# Plot
plt.plot(x, y, color='green', label=f'y = {k}x + {n}')


In [ ]:
# MLP model: single linear neuron for regression analysis

# Create some input / output data
xs = [0.03, 0.19, 0.34, 0.46, 0.78, 0.81, 1.08, 1.18, 1.39, 1.60, 1.65, 1.90]
ys = [0.67, 0.85, 1.05, 1.0, 1.40, 1.5, 1.3, 1.54, 1.55, 1.68, 1.73, 1.6 ]

print(xs)
print(ys)

model = MLP(1, [1])
print(model)
print("number of parameters", len(model.parameters()))

In [ ]:
Xb, yb = xs, ys
inputs = [[Value(xrow)] for xrow in Xb]
scores = list(map(model, inputs))
scores

In [ ]:
# loss function
def loss():
    
    Xb, yb = xs, ys
    inputs = [[Value(xr)] for xr in Xb]
    
    # forward the model to get scores
    scores = list(map(model, inputs))
    
    losses = [(yi - scorei) ** 2 for yi, scorei in zip(yb, scores)]
    data_loss = sum(losses) * (1.0 / len(losses))
    # L2 regularization
    alpha = 1e-4
    reg_loss = alpha * sum((p*p for p in model.parameters()))
    total_loss = data_loss # + reg_loss
    
    return total_loss

total_loss = loss()
print(total_loss)

In [ ]:
# optimization
def optimize(learning_rate, steps = 1):
    for k in range(steps):
        
        # forward
        total_loss = loss()
        
        # backward
        model.zero_grad()
        total_loss.backward()
        
        # update (sgd)
        for p in model.parameters():
            p.data -= learning_rate * p.grad
        
        total_loss = loss() # for correct display after the total_loss.backward()
        if k % 1 == 0:
            print(f"step {k} loss {total_loss.data}")
    return total_loss.data


In [ ]:
num_epochs = 201
lr = [] # learning rates between 1 and 0.1
for i in range(num_epochs):
    lr.append(0.1 - i*0.09/num_epochs)

# training loop (call after each update)
for epoch in range(num_epochs):
    # ... train / update model ...
    print(f"Epoch {epoch}/{num_epochs-1}")
    myloss = optimize(lr[epoch], 1)

In [ ]:
summarize_mlp(model)

In [ ]:
visualize_mlp_architecture(model, filename="single_architecture")

In [ ]:
# test of animated gif
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# Create figure and axis
fig, ax = plt.subplots()
ims = []   # list to store frames

# Generate 20 frames
x = np.linspace(0, 2*np.pi, 200)
for i in range(20):
    y = np.sin(x + i * 0.3)
    # Each frame must be a list of artists
    frame = ax.plot(x, y, color='blue')
    ims.append(frame)

# Create and save the animation
ani = animation.ArtistAnimation(fig, ims, interval=100, blit=True)
ani.save("sine_wave.gif", writer='pillow')
plt.close(fig)
